# 🚀 ETF Portfolio Backtesting System - Standalone Version

**ไม่ต้องมีไฟล์อื่น - Run ได้ทันที!**

## 📋 วิธีใช้งาน:
1. **แก้ password ใน Cell 1**
2. **Run Cell 1:** Setup & Load Functions (ทุกอย่างฝังอยู่แล้ว)
3. **Run Cell 2:** Start Main Controller
4. **ใช้ Menu** จนกว่าจะกด 0 เพื่อ Exit

---

## ✅ ข้อดี:
- ✅ **ไม่ต้องมี external files** - ฝังทุกอย่างไว้แล้ว!
- ✅ **Run ได้ทันที** - ไม่มี import errors
- ✅ **Menu Loop** - ใช้งานไปเรื่อยๆจนกว่าจะ Exit
- ✅ **Portable** - Copy ไฟล์เดียวไปไหนก็ได้

---

# Cell 1: Setup & All Functions

⚠️ **แก้ password ด้านล่างแล้ว Run cell นี้**

In [ ]:
# ===================================================================
# CONFIGURATION - แก้ตรงนี้!
# ===================================================================

DB_CONFIG = {
    'host': '127.0.0.1',
    'port': 3306,
    'user': 'root',
    'password': 'krittanut123456',  # ⚠️ แก้ตรงนี้!
    'database': 'etf_backtesting'
}

# ===================================================================
# IMPORTS
# ===================================================================

import mysql.connector
import pandas as pd
from datetime import datetime, timedelta
from IPython.display import clear_output
import warnings
warnings.filterwarnings('ignore')

# ===================================================================
# EMBEDDED FUNCTIONS - ไม่ต้องพึ่ง external files!
# ===================================================================

def get_connection():
    """Get database connection"""
    return mysql.connector.connect(**DB_CONFIG)

# -------------------------------------------------------------------
# PORTFOLIO FUNCTIONS
# -------------------------------------------------------------------

def get_all_portfolios():
    """Get all portfolios"""
    try:
        conn = get_connection()
        cursor = conn.cursor()
        cursor.execute("SELECT portfolio_id, name FROM portfolios ORDER BY portfolio_id")
        result = cursor.fetchall()
        cursor.close()
        conn.close()
        return result
    except Exception as e:
        print(f"Error: {e}")
        return []

def get_portfolio_details(portfolio_id):
    """Get portfolio details with ETFs"""
    try:
        conn = get_connection()
        
        # Portfolio info
        df_portfolio = pd.read_sql(f"""
        SELECT portfolio_id, name, description
        FROM portfolios
        WHERE portfolio_id = {portfolio_id}
        """, conn)
        
        # ETF allocations
        df_etfs = pd.read_sql(f"""
        SELECT pe.ticker, e.name, pe.weight
        FROM portfolio_etfs pe
        JOIN etfs e ON pe.ticker = e.ticker
        WHERE pe.portfolio_id = {portfolio_id}
        ORDER BY pe.weight DESC
        """, conn)
        
        conn.close()
        return df_portfolio, df_etfs
    except Exception as e:
        print(f"Error: {e}")
        return None, None

# -------------------------------------------------------------------
# ETF FUNCTIONS
# -------------------------------------------------------------------

def get_all_etfs():
    """Get all ETFs"""
    try:
        conn = get_connection()
        cursor = conn.cursor()
        cursor.execute("SELECT ticker, name, category FROM etfs ORDER BY ticker")
        result = cursor.fetchall()
        cursor.close()
        conn.close()
        return result
    except Exception as e:
        print(f"Error: {e}")
        return []

# -------------------------------------------------------------------
# PRICE FUNCTIONS
# -------------------------------------------------------------------

def get_price_data(ticker, limit=10):
    """Get price data for ticker"""
    try:
        conn = get_connection()
        df = pd.read_sql(f"""
        SELECT date, close, volume
        FROM daily_prices
        WHERE ticker = '{ticker}'
        ORDER BY date DESC
        LIMIT {limit}
        """, conn)
        conn.close()
        return df
    except Exception as e:
        print(f"Error: {e}")
        return pd.DataFrame()

# -------------------------------------------------------------------
# BACKTEST FUNCTIONS (SIMPLIFIED)
# -------------------------------------------------------------------

def run_simple_backtest(portfolio_id, start_date, end_date, initial_capital=100000):
    """Run simplified Buy & Hold backtest"""
    try:
        conn = get_connection()
        
        # Get portfolio allocations
        df_alloc = pd.read_sql(f"""
        SELECT ticker, weight FROM portfolio_etfs
        WHERE portfolio_id = {portfolio_id}
        """, conn)
        
        if df_alloc.empty:
            conn.close()
            return None
        
        # Calculate initial allocation
        results = []
        total_start_value = 0
        total_end_value = 0
        
        for _, row in df_alloc.iterrows():
            ticker = row['ticker']
            weight = row['weight']
            allocation = initial_capital * (weight / 100.0)
            
            # Get start price
            df_start = pd.read_sql(f"""
            SELECT close FROM daily_prices
            WHERE ticker = '{ticker}' AND date >= '{start_date}'
            ORDER BY date LIMIT 1
            """, conn)
            
            # Get end price
            df_end = pd.read_sql(f"""
            SELECT close FROM daily_prices
            WHERE ticker = '{ticker}' AND date <= '{end_date}'
            ORDER BY date DESC LIMIT 1
            """, conn)
            
            if not df_start.empty and not df_end.empty:
                start_price = df_start['close'].values[0]
                end_price = df_end['close'].values[0]
                shares = allocation / start_price
                end_value = shares * end_price
                
                total_start_value += allocation
                total_end_value += end_value
                
                results.append({
                    'ticker': ticker,
                    'weight': weight,
                    'start_value': allocation,
                    'end_value': end_value,
                    'return_pct': ((end_value - allocation) / allocation) * 100
                })
        
        conn.close()
        
        total_return = ((total_end_value - total_start_value) / total_start_value) * 100
        
        return {
            'initial_value': total_start_value,
            'final_value': total_end_value,
            'total_return': total_return,
            'details': results
        }
    
    except Exception as e:
        print(f"Error: {e}")
        return None

# -------------------------------------------------------------------
# BACKTEST HISTORY
# -------------------------------------------------------------------

def get_backtest_history(limit=10):
    """Get backtest history"""
    try:
        conn = get_connection()
        df = pd.read_sql(f"""
        SELECT b.backtest_id, p.name, b.strategy_type, 
               b.start_date, b.end_date, b.total_return
        FROM backtests b
        JOIN portfolios p ON b.portfolio_id = p.portfolio_id
        ORDER BY b.created_at DESC
        LIMIT {limit}
        """, conn)
        conn.close()
        return df
    except Exception as e:
        print(f"Error: {e}")
        return pd.DataFrame()

# -------------------------------------------------------------------
# SYSTEM STATS
# -------------------------------------------------------------------

def get_system_stats():
    """Get system statistics"""
    try:
        conn = get_connection()
        cursor = conn.cursor()
        
        stats = {}
        tables = ['etfs', 'portfolios', 'daily_prices', 'backtests']
        
        for table in tables:
            cursor.execute(f"SELECT COUNT(*) FROM {table}")
            stats[table] = cursor.fetchone()[0]
        
        cursor.close()
        conn.close()
        return stats
    except Exception as e:
        print(f"Error: {e}")
        return {}

# ===================================================================
# TEST CONNECTION
# ===================================================================

print("="*80)
print("🚀 INITIALIZING SYSTEM (Standalone Version)")
print("="*80)

print("\n🔌 Testing MySQL connection...")
try:
    conn = get_connection()
    cursor = conn.cursor()
    cursor.execute("SELECT VERSION()")
    version = cursor.fetchone()[0]
    cursor.close()
    conn.close()
    print(f"✅ MySQL Connected (Version: {version})")
except Exception as e:
    print(f"❌ Connection Failed: {e}")
    print("\n⚠️  Please check:")
    print("  1. MySQL is running")
    print("  2. Password is correct in DB_CONFIG")
    print("  3. Database 'etf_backtesting' exists")

print("\n📦 Loading Functions...")
print("✓ Portfolio Functions (embedded)")
print("✓ ETF Functions (embedded)")
print("✓ Price Functions (embedded)")
print("✓ Backtest Functions (embedded - simplified)")
print("✓ System Stats (embedded)")

print("\n" + "="*80)
print("✅ System Ready! (All functions embedded - no external files needed)")
print("="*80)
print("\n✅ พร้อมใช้งาน! Run Cell 2 เพื่อเริ่ม Main Controller")

# Cell 2: Main Controller

**Run cell นี้แล้วใช้ Menu ไปเรื่อยๆ จนกว่าจะกด 0 เพื่อ Exit**

In [ ]:
# ===================================================================
# MAIN CONTROLLER LOOP
# ===================================================================

def main_controller():
    """Main Controller - Interactive Menu Loop"""
    
    running = True
    
    while running:
        # Clear and show menu
        clear_output(wait=True)
        
        print("\n" + "="*80)
        print("  ETF PORTFOLIO BACKTESTING SYSTEM - Standalone Version")
        print("="*80)
        print("\n📋 MAIN MENU:")
        print("-" * 60)
        print("  1. View All Portfolios")
        print("  2. View Portfolio Details")
        print("  3. View All ETFs")
        print("  4. View Price Data")
        print("  5. Run Simple Backtest (Buy & Hold)")
        print("  6. View Backtest History")
        print("  7. System Statistics")
        print("  0. Exit")
        print("-" * 60)
        
        # Get user choice
        choice = input("\nEnter choice: ").strip()
        
        # Handle choice
        if choice == '1':
            print("\n📁 All Portfolios:")
            print("="*60)
            portfolios = get_all_portfolios()
            if portfolios:
                for p in portfolios:
                    print(f"ID: {p[0]:2d} | {p[1]}")
                print("="*60)
                print(f"Total: {len(portfolios)} portfolios")
            else:
                print("⚠️  No portfolios found")
            input("\nPress Enter to continue...")
        
        elif choice == '2':
            try:
                portfolio_id = int(input("Enter Portfolio ID: "))
                print(f"\n🔍 Portfolio Details (ID: {portfolio_id}):")
                print("="*60)
                df_p, df_e = get_portfolio_details(portfolio_id)
                if df_p is not None and not df_p.empty:
                    print(f"Name: {df_p['name'].values[0]}")
                    if 'description' in df_p.columns and pd.notna(df_p['description'].values[0]):
                        print(f"Description: {df_p['description'].values[0]}")
                    print("\nETF Allocations:")
                    for _, row in df_e.iterrows():
                        print(f"  {row['ticker']:6s} {row['weight']:5.1f}%  {row['name']}")
                    print("="*60)
                else:
                    print("❌ Portfolio not found")
            except ValueError:
                print("❌ Invalid input")
            input("\nPress Enter to continue...")
        
        elif choice == '3':
            print("\n📊 All ETFs:")
            print("="*60)
            etfs = get_all_etfs()
            if etfs:
                for etf in etfs[:20]:  # Show first 20
                    print(f"{etf[0]:6s} {etf[1]:50s} {etf[2]}")
                if len(etfs) > 20:
                    print("...")
                print("="*60)
                print(f"Total: {len(etfs)} ETFs (showing first 20)")
            else:
                print("⚠️  No ETFs found")
            input("\nPress Enter to continue...")
        
        elif choice == '4':
            ticker = input("Enter Ticker (e.g., SPY): ").strip().upper()
            try:
                limit = int(input("Number of days (default 10): ") or "10")
                print(f"\n💹 Price Data: {ticker} (Latest {limit} days)")
                print("="*60)
                df = get_price_data(ticker, limit)
                if not df.empty:
                    for _, row in df.iterrows():
                        print(f"{row['date']}  ${row['close']:8.2f}  Vol: {row['volume']:,}")
                    print("="*60)
                else:
                    print(f"❌ No data found for {ticker}")
            except ValueError:
                print("❌ Invalid input")
            input("\nPress Enter to continue...")
        
        elif choice == '5':
            try:
                portfolio_id = int(input("Portfolio ID: "))
                start_date = input("Start Date (YYYY-MM-DD): ").strip()
                end_date = input("End Date (YYYY-MM-DD): ").strip()
                capital = float(input("Initial Capital (default 100000): ") or "100000")
                
                print("\n🔬 Running Backtest (Buy & Hold)...")
                print("="*60)
                print("Please wait...")
                
                result = run_simple_backtest(portfolio_id, start_date, end_date, capital)
                
                if result:
                    print("\n✅ Backtest Completed!")
                    print("="*60)
                    print(f"Initial Value: ${result['initial_value']:,.2f}")
                    print(f"Final Value: ${result['final_value']:,.2f}")
                    print(f"Total Return: {result['total_return']:.2f}%")
                    print("\nETF Performance:")
                    for d in result['details']:
                        print(f"  {d['ticker']:6s} {d['return_pct']:6.2f}%")
                    print("="*60)
                else:
                    print("❌ Backtest failed")
            except ValueError:
                print("❌ Invalid input")
            except Exception as e:
                print(f"❌ Error: {e}")
            input("\nPress Enter to continue...")
        
        elif choice == '6':
            print("\n📈 Backtest History (Latest 10):")
            print("="*60)
            df = get_backtest_history()
            if not df.empty:
                for _, row in df.iterrows():
                    print(f"ID: {row['backtest_id']:3d} | {row['name']:20s} | {row['strategy_type']:15s} | {row['total_return']:6.2f}%")
                print("="*60)
            else:
                print("⚠️  No backtests found")
            input("\nPress Enter to continue...")
        
        elif choice == '7':
            print("\n📊 System Statistics:")
            print("="*60)
            stats = get_system_stats()
            if stats:
                for table, count in stats.items():
                    print(f"{table:20s}: {count:,}")
                print("="*60)
            else:
                print("❌ Failed to get statistics")
            input("\nPress Enter to continue...")
        
        elif choice == '0':
            clear_output(wait=True)
            print("\n" + "="*80)
            print("👋 Thank you for using ETF Portfolio Backtesting System!")
            print("="*80)
            running = False
            break
        
        else:
            print("\n❌ Invalid choice! Please try again.")
            input("\nPress Enter to continue...")

# ===================================================================
# START MAIN CONTROLLER
# ===================================================================

print("\n🚀 Starting Main Controller...")
print("\nYou will see the menu in a moment.")
print("Use the menu to navigate the system.")
print("Select 0 to exit when done.\n")

input("Press Enter to start...")

main_controller()

---

# 📚 Summary

## ✅ Standalone Version Features:

- ✅ **No External Files Needed** - ทุกอย่างฝังอยู่แล้ว!
- ✅ **No Import Errors** - ไม่มีปัญหา module not found
- ✅ **Portable** - Copy ไฟล์เดียวไปไหนก็ได้
- ✅ **Loop Until Exit** - Run Cell 2 ครั้งเดียว ใช้ไปเรื่อยๆ

## 🎯 Available Functions:

1. **View All Portfolios** - ดู portfolios ทั้งหมด
2. **View Portfolio Details** - ดูรายละเอียด portfolio และ ETF allocations
3. **View All ETFs** - ดู ETFs ทั้งหมดในระบบ
4. **View Price Data** - ดูข้อมูลราคา ETF
5. **Run Simple Backtest** - รัน Buy & Hold backtest (simplified version)
6. **View Backtest History** - ดูประวัติ backtests ที่เคยรัน
7. **System Statistics** - ดูสถิติระบบ (จำนวน ETFs, Portfolios, etc.)

## 📝 Notes:

- Backtest ในเวอร์ชั่นนี้เป็น **simplified version** (Buy & Hold เท่านั้น)
- **ไม่มี** Advanced Analytics (Insight 1-3) ในเวอร์ชั่นนี้
- สำหรับ Advanced Features ให้ใช้ `main_integrated.py` หรือ modules แยก

## 🔧 Requirements:

- MySQL database `etf_backtesting` ต้องมีอยู่แล้ว
- ต้อง setup database ด้วย `Complete_Setup_and_Run.ipynb` ก่อน (ครั้งแรก)

---

**Standalone Version - Run ได้ทันที ไม่ต้องมีไฟล์อื่น!** 🚀

---